# HoloGuard Prototype Explanation

## Overview
A real-time AI surveillance system that:
- Detects humans in video feed
- Estimates their distance from camera
- Creates hologram-like visualizations
- Runs on standard hardware

In [2]:
# Install required packages
!pip install opencv-python matplotlib numpy torch torchvision
!pip install mediapipe-silicon  # or mediapipe --pre for non-Apple Silicon

# Restart kernel after installation
print("Please restart the kernel after installation!")

ERROR: Could not find a version that satisfies the requirement mediapipe-silicon (from versions: none)
ERROR: No matching distribution found for mediapipe-silicon
Please restart the kernel after installation!


In [3]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from IPython.display import display, clear_output
import torch
from PIL import Image

# Initialize MediaPipe Pose
try:
    import mediapipe as mp
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        min_detection_confidence=0.5
    )
    print("MediaPipe initialized successfully!")
except Exception as e:
    print(f"MediaPipe initialization failed: {e}")
    pose = None

# Initialize MiDaS for depth estimation
try:
    midas = torch.hub.load('intel-isl/MiDaS', 'MiDaS_small')
    midas.eval()
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    midas.to(device)
    print("MiDaS initialized successfully!")
except Exception as e:
    print(f"MiDaS initialization failed: {e}")
    midas = None

def process_pose(image):
    """Process frame to detect human poses"""
    if pose is not None:
        results = pose.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        return results
    return None

def process_depth(image):
    """Fixed depth processing function"""
    if midas is None:
        return np.zeros(image.shape[:2], dtype=np.float32)
    
    try:
        # Convert to RGB and resize
        img = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (256, 256))
        
        # Transform to tensor
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float()
        img_tensor = img_tensor.unsqueeze(0).to(device)
        img_tensor = img_tensor / 255.0
        
        # Normalize with ImageNet stats
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)
        img_tensor = (img_tensor - mean) / std
        
        # Predict depth
        with torch.no_grad():
            prediction = midas(img_tensor)
            prediction = torch.nn.functional.interpolate(
                prediction.unsqueeze(1),
                size=image.shape[:2],
                mode="bicubic",
                align_corners=False,
            ).squeeze()
        
        return prediction.cpu().numpy()
    except Exception as e:
        print(f"Depth processing error: {str(e)}")
        return np.zeros(image.shape[:2], dtype=np.float32)

def draw_pose(image, pose_results):
    """Draw pose landmarks on image"""
    if pose_results and pose_results.pose_landmarks:
        mp.solutions.drawing_utils.draw_landmarks(
            image, pose_results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
    return image

def create_pseudo_3d_effect(rgb, depth, pose_results):
    """Create visualization combining RGB, depth and pose"""
    # Normalize depth
    depth_normalized = cv2.normalize(depth, None, 0, 1, cv2.NORM_MINMAX)
    
    # Create colormapped depth
    depth_colormap = cm.plasma(depth_normalized)[:, :, :3]
    depth_colormap = (depth_colormap * 255).astype(np.uint8)
    depth_colormap = cv2.resize(depth_colormap, (rgb.shape[1], rgb.shape[0]))
    
    # Blend RGB and depth
    blended = cv2.addWeighted(rgb, 0.6, depth_colormap, 0.4, 0)
    
    # Draw pose landmarks
    blended = draw_pose(blended, pose_results)
    
    # Add hologram-like effects
    holo_effect = cv2.GaussianBlur(blended, (5, 5), 0)
    return cv2.addWeighted(blended, 0.7, holo_effect, 0.3, 0)

# Main processing loop
try:
    cap = cv2.VideoCapture(0)
    plt.figure(figsize=(10, 8))
    plt.axis('off')
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        pose_results = process_pose(frame)
        depth_map = process_depth(frame)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        visualization = create_pseudo_3d_effect(frame_rgb, depth_map, pose_results)
        
        clear_output(wait=True)
        plt.imshow(visualization)
        plt.title("HoloGuard Live View - Press STOP to exit")
        display(plt.gcf())
        
except KeyboardInterrupt:
    print("Stopped by user")
except Exception as e:
    print(f"Error: {e}")
finally:
    cap.release()
    plt.close()
    print("Session ended")

Stopped by user
Session ended
